In [1]:
import torch
import numpy as np
import json
from torchvision import datasets, transforms

# Artifacts
qdict = torch.load("artifacts/mnist_mlp_int8_weights.pth", map_location="cpu")

np.save("artifacts/fc1_weight.npy", qdict["fc1.weight_int8"].numpy())
np.save("artifacts/fc1_bias.npy",   qdict["fc1.bias_fp32"].numpy())
np.save("artifacts/fc2_weight.npy", qdict["fc2.weight_int8"].numpy())
np.save("artifacts/fc2_bias.npy",   qdict["fc2.bias_fp32"].numpy())
np.save("artifacts/fc3_weight.npy", qdict["fc3.weight_int8"].numpy())
np.save("artifacts/fc3_bias.npy",   qdict["fc3.bias_fp32"].numpy())

weight_scales = {
    "fc1": float(qdict["fc1.weight_scale"].item()),
    "fc2": float(qdict["fc2.weight_scale"].item()),
    "fc3": float(qdict["fc3.weight_scale"].item()),
}
with open("artifacts/weight_scales.json", "w") as f:
    json.dump(weight_scales, f, indent=2)

# INT32 biases for monolithic FPGA IP (bias added in integer domain)
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

np.save("artifacts/fc1_bias_int32.npy",
    np.round(qdict["fc1.bias_fp32"].numpy() / (act_scales["input"]     * weight_scales["fc1"])).astype(np.int32))
np.save("artifacts/fc2_bias_int32.npy",
    np.round(qdict["fc2.bias_fp32"].numpy() / (act_scales["relu1_out"] * weight_scales["fc2"])).astype(np.int32))
np.save("artifacts/fc3_bias_int32.npy",
    np.round(qdict["fc3.bias_fp32"].numpy() / (act_scales["relu2_out"] * weight_scales["fc3"])).astype(np.int32))

# MNIST test set
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())
images = test_ds.data.numpy().astype(np.float32) / 255.0 # (10000, 28, 28), [0, 1]
labels = test_ds.targets.numpy() # (10000,)

np.save("artifacts/mnist_test_images.npy", images)
np.save("artifacts/mnist_test_labels.npy", labels)

print("Saved .npy artifacts")
print("Files written:")
import os
for f in sorted(os.listdir("artifacts")):
    print(f"  artifacts/{f}")


Saved .npy artifacts
Files written:
  artifacts/activation_scales.json
  artifacts/fc1_bias.npy
  artifacts/fc1_bias_int32.npy
  artifacts/fc1_weight.npy
  artifacts/fc2_bias.npy
  artifacts/fc2_bias_int32.npy
  artifacts/fc2_weight.npy
  artifacts/fc3_bias.npy
  artifacts/fc3_bias_int32.npy
  artifacts/fc3_weight.npy
  artifacts/mnist_mlp_fp32.pth
  artifacts/mnist_mlp_int8_weights.pth
  artifacts/mnist_test_images.npy
  artifacts/mnist_test_labels.npy
  artifacts/weight_scales.json
